# Математический маятник

$\ddot\theta + \omega^2\theta = 0$, $\omega = \sqrt{g/l}$.
При $v = \dot\theta$ система имеет вид $\dot\theta = v$, $\dot v = -\omega^2\theta$.

Точное решение:
$$\theta(t) = \theta_0\cos(\omega t) + \frac{v_0}{\omega}\sin(\omega t),$$
$$v(t) = -\omega\theta_0\sin(\omega t) + v_0\cos(\omega t).$$

Энергии в линейной модели:
$$e_p = \frac{ml^2\omega^2\theta^2}{2}, \qquad e_k = \frac{ml^2v^2}{2}.$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

g = 9.81
l = 1.0
m = 1.0
omega = np.sqrt(g / l)
theta_0 = 0.1
v_0 = 0.0
y_0 = np.array([theta_0, v_0])
t_end = 6.0
steps = [0.04, 0.02, 0.01]


In [ ]:
def rhs(y):  # правая часть системы
    return np.array([y[1], -omega**2 * y[0]])


def euler_step(y, h):  # шаг явного эйлера
    return y + h * rhs(y)


def rk4_step(y, h):  # шаг rk4
    k_1 = rhs(y)
    k_2 = rhs(y + h * k_1 / 2)
    k_3 = rhs(y + h * k_2 / 2)
    k_4 = rhs(y + h * k_3)
    return y + h * (k_1 + 2 * k_2 + 2 * k_3 + k_4) / 6


def solve(step, h):  # расчёт на временной сетке
    n = round(t_end / h)
    t = np.linspace(0, t_end, n + 1)
    dt = t[1] - t[0]
    y = np.empty((n + 1, 2))
    y[0] = y_0
    for i in range(n):
        y[i + 1] = step(y[i], dt)
    return t, y


def exact(t):  # точное решение
    theta = theta_0 * np.cos(omega * t) + v_0 / omega * np.sin(omega * t)
    v = -omega * theta_0 * np.sin(omega * t) + v_0 * np.cos(omega * t)
    return np.column_stack((theta, v))


def energy(y):  # потенциальная и кинетическая энергии
    e_p = m * l**2 * omega**2 * y[:, 0]**2 / 2
    e_k = m * l**2 * y[:, 1]**2 / 2
    return e_p, e_k


In [ ]:
results = []
for h in steps:
    t, y_euler = solve(euler_step, h)
    _, y_rk4 = solve(rk4_step, h)
    results.append((t, exact(t), y_euler, y_rk4))

names = ['точное', 'эйлер', 'rk4']
colors = ['black', 'tab:orange', 'tab:blue']
styles = ['-', '--', ':']

print(f"{'шаг, с':>8} {'метод':>8} {'макс. ошибка θ, рад':>22} {'изменение энергии, %':>24}")
for h, (t, y_exact, y_euler, y_rk4) in zip(steps, results):
    for name, y in [('эйлер', y_euler), ('rk4', y_rk4)]:
        error = np.max(np.abs(y[:, 0] - y_exact[:, 0]))
        e_p, e_k = energy(y)
        e = e_p + e_k
        drift = 100 * (e[-1] / e[0] - 1)
        print(f'{h:8.2f} {name:>8} {error:22.6e} {drift:24.6e}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for ax, h, (t, *solutions) in zip(axes, steps, results):
    for y, name, color, style in zip(solutions, names, colors, styles):
        ax.plot(t, y[:, 0], label=name, color=color, linestyle=style)
    ax.set(title=f'h = {h} с', xlabel='t, с', ylabel='θ, рад')
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle('угол отклонения')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for ax, h, (t, *solutions) in zip(axes, steps, results):
    for y, name, color, style in zip(solutions, names, colors, styles):
        ax.plot(y[:, 0], y[:, 1], label=name, color=color, linestyle=style)
    ax.set(title=f'h = {h} с', xlabel='θ, рад', ylabel='v, рад/с')
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle('фазовые траектории')
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7), constrained_layout=True)
for j, (h, (t, *solutions)) in enumerate(zip(steps, results)):
    for y, name, color, style in zip(solutions, names, colors, styles):
        e_p, e_k = energy(y)
        axes[0, j].plot(t, e_p, label=name, color=color, linestyle=style)
        axes[1, j].plot(t, e_k, label=name, color=color, linestyle=style)
    for i, label in enumerate(['потенциальная энергия, Дж', 'кинетическая энергия, Дж']):
        axes[i, j].set(title=f'h = {h} с', xlabel='t, с', ylabel=label)
        axes[i, j].grid(alpha=0.3)
        axes[i, j].legend()
plt.show()


При уменьшении шага ошибка обоих методов снижается. У явного эйлера полная энергия растёт, поэтому фазовая траектория раскручивается наружу. У rk4 ошибка угла при делении шага пополам уменьшается примерно в 16 раз; на графиках его решение практически совпадает с точным.

## нелинейный маятник

Теперь $\ddot\theta + \omega^2\sin\theta = 0$. Масса, длина и $g$ прежние.
Для углов $0 < \theta_0 < \pi$ и нулевой начальной скорости период равен
$$t_p(\theta_0) = \frac{4}{\omega}K\left(\sin^2\frac{\theta_0}{2}\right).$$
Здесь аргумент `scipy.special.ellipk` — параметр $k^2$. Сравним эту формулу с периодом, измеренным по численной траектории.

In [ ]:
from scipy.special import ellipk
from scipy.optimize import brentq
from matplotlib.animation import FuncAnimation as func_animation
from IPython.display import HTML as html, display

In [ ]:
def model_rhs(y, omega_sq, nonlinear=True):  # правая часть модели
    angle = np.sin(y[..., 0]) if nonlinear else y[..., 0]
    return np.stack((y[..., 1], -omega_sq * angle), axis=-1)


def solve_model(theta_start, v_start, h, t_stop, omega_sq, nonlinear=True):  # rk4 для одного маятника или ансамбля
    theta_start, v_start = np.broadcast_arrays(theta_start, v_start)
    y_start = np.stack((theta_start, v_start), axis=-1)
    n = int(np.ceil(t_stop / h))
    t = np.linspace(0, t_stop, n + 1)
    dt = t[1] - t[0]
    y = np.empty((n + 1,) + y_start.shape)
    y[0] = y_start
    for i in range(n):
        k_1 = model_rhs(y[i], omega_sq, nonlinear)
        k_2 = model_rhs(y[i] + dt * k_1 / 2, omega_sq, nonlinear)
        k_3 = model_rhs(y[i] + dt * k_2 / 2, omega_sq, nonlinear)
        k_4 = model_rhs(y[i] + dt * k_3, omega_sq, nonlinear)
        y[i + 1] = y[i] + dt * (k_1 + 2 * k_2 + 2 * k_3 + k_4) / 6
    return t, y


def measure_period(t, theta):  # период по пересечениям нуля сверху вниз
    indices = np.flatnonzero((theta[:-1] > 0) & (theta[1:] <= 0))
    if len(indices) < 2:
        raise ValueError('нужны хотя бы два пересечения нуля в одну сторону')
    crossings = t[indices] + (t[indices + 1] - t[indices]) * theta[indices] / (
        theta[indices] - theta[indices + 1]
    )
    return np.mean(np.diff(crossings))


def exact_period(amplitude):  # период нелинейного маятника
    return 4 / omega * ellipk(np.sin(amplitude / 2)**2)

In [ ]:
amplitudes_deg = np.array([1, 5, 10, 15, 20, 25, 30, 45, 60, 90, 120, 150, 170])
amplitudes = np.deg2rad(amplitudes_deg)
t_period, y_period = solve_model(amplitudes, 0.0, 0.002, 32.0, omega**2)
periods = np.array([
    measure_period(t_period, y_period[:, j, 0]) for j in range(len(amplitudes))
])
periods_exact = exact_period(amplitudes)
period_linear = 2 * np.pi / omega
period_error = (periods_exact - period_linear) / periods_exact

print(f"{'амплитуда, °':>13} {'период rk4, с':>15} {'точный, с':>12} {'ошибка лин., %':>16}")
for angle, numeric, reference, error in zip(amplitudes_deg, periods, periods_exact, period_error):
    print(f'{angle:13.0f} {numeric:15.6f} {reference:12.6f} {100 * error:16.3f}')
print(f'макс. относительная ошибка численного периода: {np.max(np.abs(periods / periods_exact - 1)):.2e}')

angle_grid = np.linspace(0, 170, 500)
period_grid = exact_period(np.deg2rad(angle_grid))
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(angle_grid, period_grid, color='black', label='точная формула')
axes[0].plot(amplitudes_deg, periods, 'o', color='tab:blue', markersize=4, label='rk4')
axes[0].axhline(period_linear, color='tab:orange', linestyle='--', label='линейная модель')
axes[0].set(xlabel='начальная амплитуда, °', ylabel='период, с')
axes[1].plot(angle_grid, 100 * (1 - period_linear / period_grid), color='tab:blue', label='ошибка периода')
for tolerance in [0.01, 0.05]:
    limit = brentq(lambda a: 1 - period_linear / exact_period(a) - tolerance, 0, np.pi - 0.001)
    print(f'ошибка до {100 * tolerance:.0f}%: амплитуда до {np.rad2deg(limit):.2f}°')
    axes[1].axhline(100 * tolerance, linestyle='--', alpha=0.6, label=f'{100 * tolerance:.0f}%')
    axes[1].plot(np.rad2deg(limit), 100 * tolerance, 'ko', markersize=4)
axes[1].set(xlabel='начальная амплитуда, °', ylabel='ошибка линейного периода, %')
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
plt.show()

Ошибка линейного периода считается относительно нелинейного: $\varepsilon = (t_p-t_\mathrm{лин})/t_p$.
Линейная модель занижает период. При нулевой начальной скорости ошибка не превышает 1% до 22.93° и 5% до 51.35°. Границы рассчитаны по точной формуле.

### расхождение моделей при 60°

Сравниваем нелинейное решение с точным линейным при одинаковом начальном угле и нулевой скорости. Шаг нелинейного расчёта — 0.0005 с.

In [ ]:
amplitude = np.deg2rad(60)
h_ref = 0.0005
t_compare = 12.0
t_ref, y_ref = solve_model(amplitude, 0.0, h_ref, t_compare, omega**2)
y_linear = np.column_stack((
    amplitude * np.cos(omega * t_ref),
    -amplitude * omega * np.sin(omega * t_ref)
))
model_difference = y_ref[:, 0] - y_linear[:, 0]
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True, constrained_layout=True)
axes[0].plot(t_ref, y_ref[:, 0], color='tab:blue', label='нелинейная')
axes[0].plot(t_ref, y_linear[:, 0], '--', color='tab:orange', label='линейная')
axes[0].set(ylabel='θ, рад', title='начальная амплитуда 60°')
axes[0].legend()
axes[1].plot(t_ref, model_difference, color='tab:blue')
axes[1].set(xlabel='t, с', ylabel='θ нелин. − θ лин., рад')
for ax in axes:
    ax.grid(alpha=0.25)
plt.show()
print(f'линейный период: {period_linear:.6f} с')
print(f'нелинейный период: {exact_period(amplitude):.6f} с')
print(f'ошибка линейного периода: {100 * (1 - period_linear / exact_period(amplitude)):.3f}%')

### уменьшение шага при 60°

Здесь сравниваются только нелинейные решения rk4. Эталон — расчёт с шагом 0.0005 с; ошибка угла измеряется в общих узлах сеток на интервале 0–12 с.

In [ ]:
check_steps = np.array([0.04, 0.02, 0.01, 0.005])
step_errors = []
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for h in check_steps:
    t, y = solve_model(amplitude, 0.0, h, t_compare, omega**2)
    indices = np.rint(t / h_ref).astype(int)
    difference = y[:, 0] - y_ref[indices, 0]
    step_errors.append(np.max(np.abs(difference)))
    axes[0].plot(t, difference, label=f'h = {h} с')
step_errors = np.array(step_errors)
axes[0].set(xlabel='t, с', ylabel='θ − θ эталон, рад')
axes[0].ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
axes[1].loglog(check_steps, step_errors, 'o-', label='макс. ошибка')
axes[1].loglog(check_steps, step_errors[0] * (check_steps / check_steps[0])**4, '--', label='пропорционально h⁴')
axes[1].set(xlabel='шаг, с', ylabel='макс. ошибка угла, рад')
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
plt.show()
print(f"{'шаг, с':>10} {'макс. ошибка, рад':>20} {'порядок':>10}")
for i, (h, error) in enumerate(zip(check_steps, step_errors)):
    order = '—' if i == 0 else f'{np.log2(step_errors[i - 1] / error):.2f}'
    print(f'{h:10.3f} {error:20.6e} {order:>10}')

При уменьшении шага оценка порядка растёт от 3.72 до 3.94, приближаясь к четвёртому порядку rk4. Расхождение линейной и нелинейной моделей связано с разными периодами; уменьшение шага его не устраняет.

### линейный и нелинейный маятник

Слева — положения маятников, справа — синхронные точки и их фазовые следы. Начальная амплитуда 60°, длина 1 м. Скорость воспроизведения 1× соответствует времени модели.

In [ ]:
def animate_comparison(t, nonlinear_y, linear_y, stride=100):  # маятники и синхронные фазовые следы
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.8), dpi=80, constrained_layout=True)
    axes[0].set(xlim=(-1.15 * l, 1.15 * l), ylim=(-1.15 * l, 0.25 * l), xlabel='x, м', ylabel='y, м')
    axes[0].set_aspect('equal')
    axes[0].plot(0, 0, 'ko', markersize=4)
    axes[1].set(xlim=(-1.2 * amplitude, 1.2 * amplitude), ylim=(-1.2 * omega * amplitude, 1.2 * omega * amplitude), xlabel='θ, рад', ylabel='v, рад/с')
    rods, traces, dots = [], [], []
    for color, style, name in [('tab:blue', '-', 'нелинейная'), ('tab:orange', '--', 'линейная')]:
        rod, = axes[0].plot([], [], linestyle=style, color=color, marker='o', markevery=[1], label=name)
        trace, = axes[1].plot([], [], linestyle=style, color=color, linewidth=1.2)
        dot, = axes[1].plot([], [], 'o', color=color, markersize=5)
        rods.append(rod)
        traces.append(trace)
        dots.append(dot)
    axes[0].legend(loc='upper right', fontsize=8)
    for ax in axes:
        ax.grid(alpha=0.2)
    title = fig.suptitle('t = 0.00 с')

    def update(i):  # обновление кадра
        for y, rod, trace, dot in zip([nonlinear_y, linear_y], rods, traces, dots):
            theta = y[i, 0]
            rod.set_data([0, l * np.sin(theta)], [0, -l * np.cos(theta)])
            trace.set_data(y[:i + 1, 0], y[:i + 1, 1])
            dot.set_data([theta], [y[i, 1]])
        title.set_text(f't = {t[i]:.2f} с')
        return rods + traces + dots + [title]

    anim = func_animation(fig, update, frames=range(0, len(t), stride), interval=1000 * stride * (t[1] - t[0]), blit=False)
    plt.close(fig)
    return anim


comparison_animation = animate_comparison(t_ref, y_ref, y_linear)
display(html(comparison_animation.to_jshtml()))

### разные длины, одинаковый угол

Десять независимых нелинейных маятников с общим подвесом: $l_j=l(10/j)^2$, $j=10,\ldots,19$.
Начальный угол у всех 30°, скорость нулевая. При одинаковой амплитуде период пропорционален $\sqrt{l_j}$, поэтому частоты соотносятся как 10:11:…:19. Здесь $f_b$ — общая базовая частота. За один общий цикл маятники расходятся по фазе и возвращаются к начальной конфигурации.

In [ ]:
def animate_ensemble(t, theta, lengths, labels, fps=30):  # ансамбль с общим подвесом
    frame_times = np.linspace(0, t[-1], round(t[-1] * fps) + 1)
    frame_angles = np.column_stack([
        np.interp(frame_times, t, theta[:, j]) for j in range(len(lengths))
    ])
    max_length = np.max(lengths)
    x_limit = np.max(np.abs(lengths * np.sin(theta))) + 0.15 * max_length
    y_top = max(0, np.max(-lengths * np.cos(theta))) + 0.15 * max_length
    fig, ax = plt.subplots(figsize=(6.8, 4.4), dpi=72, constrained_layout=True)
    ax.set(xlim=(-x_limit, x_limit), ylim=(-1.12 * max_length, y_top))
    ax.set_aspect('equal')
    ax.axis('off')
    ax.plot(0, 0, 'o', color='black', markersize=4, zorder=5)
    colors = plt.cm.viridis(np.linspace(0.05, 0.9, len(lengths)))
    rods = []
    for label, color in zip(labels, colors):
        rod, = ax.plot([], [], '-o', color=color, linewidth=1.2,
                       markevery=[1], markersize=6, label=label)
        rods.append(rod)
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5),
              frameon=False, fontsize=8)
    title = ax.set_title('t = 0.00 с', fontsize=11)

    def update(i):  # обновление кадра
        for j, rod in enumerate(rods):
            x = lengths[j] * np.sin(frame_angles[i, j])
            y = -lengths[j] * np.cos(frame_angles[i, j])
            rod.set_data([0, x], [0, y])
        title.set_text(f't = {frame_times[i]:.2f} с')
        return rods + [title]

    anim = func_animation(fig, update, frames=len(frame_times),
                          interval=1000 * (frame_times[1] - frame_times[0]), blit=False)
    plt.close(fig)
    return anim


multipliers = np.arange(10, 20)
omega_base = omega / 10
frequencies = multipliers * omega_base
lengths = g / frequencies**2
ensemble_amplitude = np.deg2rad(30)
return_time = 10 * exact_period(ensemble_amplitude)
t_lengths, y_lengths = solve_model(np.full(10, ensemble_amplitude), 0.0, 0.002, return_time, frequencies**2)
length_labels = [f'f/f_b = {j}; l = {length:.2f} м' for j, length in zip(multipliers, lengths)]
lengths_animation = animate_ensemble(t_lengths, y_lengths[..., 0], lengths, length_labels)
display(html(lengths_animation.to_jshtml()))


### одинаковая длина, разные углы

Десять независимых нелинейных маятников длиной 1 м подвешены в одной точке. Начальные углы от 5° до 85°, начальные скорости нулевые. Большие амплитуды дают больший период, поэтому движения постепенно расходятся по фазе.

In [ ]:
ensemble_angles = np.linspace(5, 85, 10)
t_angles, y_angles = solve_model(np.deg2rad(ensemble_angles), 0.0, 0.01, 12.0, omega**2)
angle_labels = [f'θ₀ = {angle:.1f}°' for angle in ensemble_angles]
angles_animation = animate_ensemble(t_angles, y_angles[..., 0], np.full(10, l), angle_labels)
display(html(angles_animation.to_jshtml()))